# B2.1 · Plan–act–verify

**Function B — Product & Application Security → The Security Automation / Harness Engineer**  ·  *AI for Security*

Builds on **[B2.0 · What an agentic harness actually is](https://spbreed.github.io/cyber-commons/lessons/B2.0.html)**.

| | |
|---|---|
| Open-source tooling | Python, LiteLLM |
| Open-weight models | GLM-4.6, Llama 3.3, Kimi K2 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


A harness is four moves in a loop:

1. **Plan** — the model proposes what to do next.
2. **Act** — the harness executes that proposal against a tool.
3. **Verify** — something decides whether the result is acceptable.
4. **Stop** — either verification succeeded, or a budget ran out.

That is the whole architecture. Everything that makes a harness safe or unsafe
lives in moves 3 and 4, and this track spends most of its time there.

The reason to build it explicitly rather than adopt a framework is that
frameworks make moves 1 and 2 easy and leave 3 and 4 as your problem — usually
with a default that is "the model says it's done" and "loop forever". You need
to know exactly what yours does.

This lesson builds the loop, in about forty lines, and then changes exactly one
thing — the verifier — to show that the same model and the same proposals
produce opposite outcomes.

> **About the model in this notebook.** It runs offline against a deterministic
> replay, so the lesson executes on a Kaggle kernel with no network. The replay
> is not a language model and is labelled as such. To run the identical harness
> against a real open-weight model:
>
> ```bash
> ollama pull glm-4.6            # or kimi-k2, llama3.3
> export OPENAI_BASE_URL=http://localhost:11434/v1 OPENAI_API_KEY=ollama MODEL=glm-4.6
> ```

## 2 · Demo — the loop, with a task that has a right answer

The task: fix a function that mis-computes a security-relevant value. The model produces three attempts, the last of which is correct.

In [ ]:
import time
from dataclasses import dataclass, field

class ReplayModel:
    """DETERMINISTIC REPLAY — not a language model.
    Emits a fixed sequence so the loop's control flow is the thing under test."""
    def __init__(self, proposals, name="replay"):
        self.proposals, self.name, self.calls = list(proposals), name, 0
    def propose(self, _prompt):
        # after the script runs out it repeats — exactly what a stuck loop does
        p = self.proposals[min(self.calls, len(self.proposals) - 1)]
        self.calls += 1
        return p

@dataclass
class Step:
    n: int; proposal: str; ok: bool; detail: str; ms: float = 0.0

@dataclass
class Trace:
    steps: list = field(default_factory=list)
    stopped_by: str = ""
    succeeded: bool = False
    def table(self):
        rows = [f"{'step':>4}  {'ok':<6}{'proposal':<44}detail",
                f"{'-'*4}  {'-'*6}{'-'*44}{'-'*28}"]
        for s in self.steps:
            rows.append(f"{s.n:>4}  {str(s.ok):<6}{s.proposal[:44]:<44}{s.detail[:28]}")
        rows.append(f"\nstopped by: {self.stopped_by}    succeeded: {self.succeeded}")
        return "\n".join(rows)

def run(model, verifier, goal="", max_steps=5, max_seconds=10.0):
    tr, started = Trace(), time.monotonic()
    for n in range(1, max_steps + 1):
        t0 = time.monotonic()
        proposal = model.propose(f"{goal} (attempt {n})")      # PLAN
        ok, detail = verifier(proposal)                        # ACT + VERIFY
        tr.steps.append(Step(n, proposal, ok, detail, (time.monotonic()-t0)*1000))
        if ok:                                                 # STOP
            tr.stopped_by, tr.succeeded = "verifier satisfied", True
            return tr
        if time.monotonic() - started > max_seconds:
            tr.stopped_by = f"time budget ({max_seconds}s)"
            return tr
    tr.stopped_by = f"step budget ({max_steps} steps)"
    return tr

ATTEMPTS = [
 "def is_expired(cert): return cert.days_left < 0",     # off-by-one: 0 is expired
 "def is_expired(cert): return cert.days_left <= 0",    # correct
]

In [ ]:
# The verifier: execute the proposal against known-good cases.
CASES = [(-5, True), (0, True), (1, False), (30, False)]

class Cert:
    def __init__(self, d): self.days_left = d

def behavioural_verifier(src):
    ns = {}
    try:
        exec(compile(src, "<proposal>", "exec"), ns)
        fn = ns["is_expired"]
    except Exception as e:
        return False, f"did not compile: {type(e).__name__}"
    for days, expected in CASES:
        got = fn(Cert(days))
        if got != expected:
            return False, f"is_expired(days_left={days}) → {got}, want {expected}"
    return True, f"all {len(CASES)} cases pass"

tr = run(ReplayModel(ATTEMPTS), behavioural_verifier,
         goal="fix certificate expiry check", max_steps=5)
print(tr.table())

## 3 · Where it breaks — change only the verifier

Same model. Same proposals. Same order. The only difference is what the loop believes when it decides it has succeeded.

In [ ]:
def self_grading_verifier(src):
    """The model judges its own work. Ships in a lot of harnesses."""
    looks_done = bool(src.strip()) and src.strip().startswith("def ")
    return looks_done, "judge: looks like a valid fix, approving"

tr2 = run(ReplayModel(ATTEMPTS), self_grading_verifier,
          goal="fix certificate expiry check", max_steps=5)
print(tr2.table())

ns = {}; exec(compile(tr2.steps[-1].proposal, "<x>", "exec"), ns)
print(f"\nthe accepted code says a cert with 0 days left is expired: "
      f"{ns['is_expired'](Cert(0))}")
print("It is not. A certificate expiring today is still valid today, and this")
print("harness just shipped that. The trace above is clean and reports success.")

## 4 · The control — the verifier is a security control

State it plainly, because the rest of the track depends on it: **the verifier decides what the harness is allowed to believe.** A harness with a weak verifier does not fail loudly. It succeeds incorrectly, produces a clean trace, and the failure is discovered downstream.

In [ ]:
def compare(model_factory, verifiers, **kw):
    out = {}
    for name, v in verifiers.items():
        tr = run(model_factory(), v, **kw)
        out[name] = {"succeeded": tr.succeeded, "steps": len(tr.steps),
                     "stopped_by": tr.stopped_by,
                     "accepted": tr.steps[-1].proposal if tr.succeeded else None}
    return out

def no_verifier(_src):
    return False, "no verifier configured"

results = compare(lambda: ReplayModel(ATTEMPTS),
                  {"behavioural (executes the code)": behavioural_verifier,
                   "self-grading (asks the model)":   self_grading_verifier,
                   "none":                            no_verifier},
                  goal="fix expiry check", max_steps=4)
print(f"{'verifier':34s}{'succeeded':11s}{'steps':7s}stopped by")
print("-" * 76)
for name, r in results.items():
    print(f"{name:34s}{str(r['succeeded']):11s}{r['steps']:<7}{r['stopped_by']}")

print("\nwhat each one accepted:")
for name, r in results.items():
    print(f"   {name:34s}{r['accepted'] or '—'}")
assert results["behavioural (executes the code)"]["accepted"] == ATTEMPTS[1]
assert results["self-grading (asks the model)"]["accepted"] == ATTEMPTS[0]

## What you just proved

The behavioural verifier rejects the off-by-one on attempt 1 and accepts the correct version on attempt 2. The self-grading verifier stops on attempt 1 and reports success, having accepted code that says a certificate with 0 days left is expired. The no-verifier run consumes the full step budget.

## Your turn

Take the harness you actually run and answer one question: what exactly does it check before it reports success? If the answer is "the model said it was done" or "the command exited 0", B2.2 is the next lesson and you need it.

---

**Next → [B2.2 · Verify signals that don't lie](https://spbreed.github.io/cyber-commons/lessons/B2.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*